# Урок 08 - Шаблон проектування мультиагентів


## Налаштування


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Чому мультиагентні системи?

Завдання у реальному світі, як планування подорожі, включають багато різних видів експертизи — логістику, місцеві знання, бюджетування та інше. Один агент, який намагається впоратися з усім, швидко стає неефективним.

Мультиагентні системи вирішують це через **спеціалізацію**: кожен агент зосереджується на одній сфері експертизи, створюючи результати вищої якості, ніж універсаліст. Вони також покращують **масштабованість** — можна додавати нових агентів (наприклад, фахівця з авіаперельотів, критика ресторанів) без переписування існуючого робочого процесу. Агенти взаємодіють через структурований конвеєр, передаючи контекст від одного до наступного.


## Створення спеціалізованих агентів


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Створення послідовного робочого процесу

`WorkflowBuilder` дозволяє зв’язувати агентів у орієнтований граф. Тут ми створюємо простий двокроковий конвеєр: **TravelPlanner** складає план подорожі, а **TravelConcierge** переглядає та покращує його.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Додавання більше агентів до робочого процесу

Однією з найбільших переваг багатофункційного шаблону є його легкість у розширенні. Нижче ми додаємо агента **BudgetReviewer**, який перевіряє план на відповідність бюджету мандрівника, відмічає пункти, які можуть перевищити ліміт витрат, та пропонує альтернативи для економії коштів. Робочий процес тепер виконує три агенти послідовно:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Підсумок

У цьому уроці ви дізналися, як:

1. **Створювати спеціалізованих агентів** — кожен зі сфокусованою роллю (планування, консьєрж, перегляд бюджету).
2. **Підключати агентів у послідовний робочий процес** за допомогою `WorkflowBuilder` та `add_edge`.
3. **Потоково передавати вихідні дані** з багатопроцесного конвеєра, відслідковуючи, який агент говорить.
4. **Розширювати робочий процес**, додаючи нових агентів у ланцюг без зміни існуючих.

Шаблон багатозадачного дизайну зберігає простоту кожного агента, забезпечуючи при цьому більш багатогранні та ретельніше перевірені результати, ніж міг би досягти будь-який один агент самостійно.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:  
Цей документ було перекладено за допомогою сервісу автоматичного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, враховуйте, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ мовою оригіналу слід вважати авторитетним джерелом. Для критично важливої інформації рекомендовано звертатися до професійного перекладу, виконаного людиною. Ми не несемо відповідальності за будь-які непорозуміння чи неправильне тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
